# 🏛️ UIT Legal IR — 2-Stage Retrieval & Re-ranking (Chuẩn BTC SoICT/UIT)

## Pipeline: BM25 + Multi-Dense Embeddings → PhoRanker Cross-Encoder Re-ranking
**Kiến trúc dựa trên nghiên cứu chính thức của BTC (arXiv:2507.14619v1 — Team 4Huiter, Top 3 SoICT Hackathon)**

### Datasets & Cache inputs cần thêm vào Notebook trên Kaggle:
| Dữ liệu / Cache | Đường dẫn Kaggle |
|:---|:---|
| Dữ liệu luật & câu hỏi | `/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data` |
| Mô hình Bi-Encoder đã fine-tune | `/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder` |
| Output từ Notebook trước (PKL Cache) | `/kaggle/input/notebooks/thurdayafternoon/legal-ir` |
| Cache PKL (Dataset riêng nếu có) | `/kaggle/input/datasets/thurdayafternoon/pkl-cache` |


In [ ]:
import os
import subprocess
import re

# ⚠️ Khóa về Single GPU ngay từ đầu để tránh lỗi PyTorch DataParallel ('tokenizer' attribute error trên T4x2)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Kiểm tra GPU
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).strip()
    print(f"🎮 GPU: {gpu_info}")
    cap = float(re.search(r'(\d+\.\d+)', gpu_info.split(',')[-1]).group(1))
    if cap < 7.0:
        print("⚠️ GPU compute capability < 7.0 (P100). Cài PyTorch CUDA 11.8...")
        !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
    else:
        print("✅ GPU tương thích với PyTorch CUDA mặc định.")
except Exception as e:
    print(f"ℹ️ Thông tin GPU: {e}")

# Cài đặt các thư viện cần thiết
!pip install -q sentence-transformers rank-bm25 pyvi scikit-learn matplotlib tqdm


In [ ]:
import os
import shutil

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# === 1. CLONE CODE TỪ GITHUB ===
REPO_URL = "https://github.com/manh123-chatgpt/implement-pp1.git"
CLONE_DIR = os.path.join(WORK_DIR, "code")

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

print("📦 Đang clone code từ GitHub...")
os.system(f"git clone {REPO_URL} {CLONE_DIR}")

# Copy toàn bộ file .py từ repo vào working dir
if os.path.exists(CLONE_DIR):
    for f in os.listdir(CLONE_DIR):
        if f.endswith(".py"):
            src = os.path.join(CLONE_DIR, f)
            dst = os.path.join(WORK_DIR, f)
            shutil.copy2(src, dst)
            print(f"  ✅ {f}")

# === 2. CẤU HÌNH ĐƯỜNG DẪN KAGGLE DATASETS & NOTEBOOK OUTPUTS ===
KAGGLE_DATA_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data"
KAGGLE_MODEL_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder"

# Danh sách các thư mục nguồn để tìm kiếm file cache PKL và outputs
KAGGLE_CACHE_DIRS = [
    "/kaggle/input/notebooks/thurdayafternoon/legal-ir",       # Output từ notebook trước
    "/kaggle/input/datasets/thurdayafternoon/pkl-cache",       # Dataset upload riêng
]

# === 3. COPY / SYMLINK DỮ LIỆU VÀO WORKING DIR ===

# 3a. Copy legal_corpus_resolved.json
RESOLVED_SRC = os.path.join(KAGGLE_DATA_DIR, "legal_corpus_resolved.json")
RESOLVED_DST = os.path.join(WORK_DIR, "legal_corpus_resolved.json")
if os.path.exists(RESOLVED_SRC) and not os.path.exists(RESOLVED_DST):
    print("📄 Đang copy legal_corpus_resolved.json...")
    shutil.copy2(RESOLVED_SRC, RESOLVED_DST)
    print(f"  ✅ {os.path.getsize(RESOLVED_DST) / 1024**2:.0f} MB")

# 3b. Symlink thư mục fine_tuned_vietnamese_bi_encoder
MODEL_LINK = os.path.join(WORK_DIR, "fine_tuned_vietnamese_bi_encoder")
if not os.path.exists(MODEL_LINK) and os.path.exists(KAGGLE_MODEL_DIR):
    os.symlink(KAGGLE_MODEL_DIR, MODEL_LINK)
    print(f"🔗 Symlink mô hình Bi-Encoder: {MODEL_LINK} → {KAGGLE_MODEL_DIR}")

# 3c. Copy/Symlink các file cache PKL (tự động dò tìm trong các nguồn)
PKL_FILES = [
    "bm25_tokenized_cache_pyvi_v2.pkl",
    "corpus_embeddings_bgem3_resolved.pkl",
    "corpus_embeddings_e5_resolved.pkl",
    "corpus_embeddings_finetuned_resolved.pkl",
    "semi_hard_negatives.pkl",
]

for pkl in PKL_FILES:
    dst = os.path.join(WORK_DIR, pkl)
    if os.path.exists(dst):
        continue
    for cache_dir in KAGGLE_CACHE_DIRS:
        src = os.path.join(cache_dir, pkl)
        if os.path.exists(src) and os.path.getsize(src) > 1024:
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1024**2
            print(f"  ✅ {pkl} ({size_mb:.0f} MB) ← {cache_dir}")
            break
    else:
        print(f"  ⚠️ {pkl} — Chưa có (sẽ tự động tính nếu cần)")

# 3d. Copy mine_semi_hard_negatives.py từ notebook output nếu có
for cache_dir in KAGGLE_CACHE_DIRS:
    src_script = os.path.join(cache_dir, "mine_semi_hard_negatives.py")
    if os.path.exists(src_script) and not os.path.exists(os.path.join(WORK_DIR, "mine_semi_hard_negatives.py")):
        shutil.copy2(src_script, os.path.join(WORK_DIR, "mine_semi_hard_negatives.py"))
        print(f"  ✅ mine_semi_hard_negatives.py ← {cache_dir}")
        break

# 3e. Copy/Symlink fine_tuned_vietnamese_cross_encoder nếu có
for cache_dir in KAGGLE_CACHE_DIRS:
    ce_src = os.path.join(cache_dir, "fine_tuned_vietnamese_cross_encoder")
    ce_dst = os.path.join(WORK_DIR, "fine_tuned_vietnamese_cross_encoder")
    if os.path.isdir(ce_src) and not os.path.exists(ce_dst):
        os.symlink(ce_src, ce_dst)
        print(f"  🔗 Symlink Cross-Encoder: {ce_dst} → {ce_src}")
        break

# === 4. TỰ ĐỘNG VÁ LỖI AN TOÀN (HOTFIX) TRONG CODE ===
# Đảm bảo 'import torch' luôn có ở đầu hybrid_retriever.py (tránh lỗi name 'torch' is not defined)
hybrid_file = os.path.join(WORK_DIR, "hybrid_retriever.py")
if os.path.exists(hybrid_file):
    with open(hybrid_file, "r", encoding="utf-8") as f:
        code = f.read()
    if "import torch" not in code[:300]:
        with open(hybrid_file, "w", encoding="utf-8") as f:
            f.write("import torch\n" + code)
        print("  🛡️ Hotfix: Đã bổ sung 'import torch' vào đầu hybrid_retriever.py")

print("\n" + "=" * 60)
print("🎯 SETUP HOÀN TẤT! Sẵn sàng chạy pipeline.")
print("=" * 60)


In [ ]:
import torch
import os

# Kiểm tra GPU
print("🖥️ THÔNG TIN HỆ THỐNG:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)")

# Kiểm tra file đã sẵn sàng
print("\n📂 KIỂM TRA FILE TRONG WORKSPACE:")
check_files = {
    "data_loader.py": "Code - Data Loader",
    "bm25_retriever.py": "Code - BM25 Searcher",
    "dense_retriever.py": "Code - Dense Retriever",
    "hybrid_retriever.py": "Code - Hybrid 2-Stage Searcher",
    "evaluator.py": "Code - Evaluator",
    "mine_semi_hard_negatives.py": "Code - Semi-Hard Negative Mining",
    "train_cross_encoder.py": "Code - Cross-Encoder Trainer",
    "legal_corpus_resolved.json": "Data - Graph Resolved Corpus",
    "fine_tuned_vietnamese_bi_encoder": "Model - Fine-tuned Bi-Encoder",
    "fine_tuned_vietnamese_cross_encoder": "Model - Fine-tuned Cross-Encoder",
    "bm25_tokenized_cache_pyvi_v2.pkl": "Cache - BM25 Tokenized",
    "corpus_embeddings_finetuned_resolved.pkl": "Cache - Bi-Encoder Embeddings",
    "corpus_embeddings_bgem3_resolved.pkl": "Cache - BGE-M3 Embeddings",
    "corpus_embeddings_e5_resolved.pkl": "Cache - E5-Large Embeddings",
    "semi_hard_negatives.pkl": "Cache - Semi-Hard Negatives",
}
for fname, desc in check_files.items():
    exists = "✅" if os.path.exists(fname) else "❌"
    size_str = f" ({os.path.getsize(fname)/1024**2:.1f} MB)" if os.path.exists(fname) and os.path.isfile(fname) else ""
    print(f"  {exists} {desc}: {fname}{size_str}")


In [ ]:
from data_loader import load_corpus, load_train_data, load_test_data

corpus = load_corpus()
train_data = load_train_data()
test_data = load_test_data()

print(f"\n📊 Corpus: {len(corpus)} văn bản pháp luật")
print(f"📊 Train: {len(train_data)} câu hỏi")
print(f"📊 Test: {len(test_data)} câu hỏi")


## 🔍 Stage 1 + Stage 2: Hybrid 2-Stage Retrieval & Re-ranking
- **Stage 1**: BM25 (Lexical) + Dense Bi-Encoders → Lọc **Top 90** ứng viên
- **Stage 2**: PhoRanker Cross-Encoder re-rank 90 ứng viên → Chọn **Top 5** chính xác nhất

> ⚡ **Tự động tối ưu chế độ:**
>- Nếu **đã có đủ 3 cache embeddings** (`finetuned`, `bgem3`, `e5`), notebook tự động chạy **Full Ensemble Mode (`light_mode=False`)** để kết hợp toàn bộ sức mạnh của 4 mô hình, cho Recall cao nhất mà chỉ mất vài giây để nạp!
>- Nếu **chưa đủ cache**, notebook tự động chạy **`light_mode=True`** (BM25 + Bi-Encoder + PhoRanker) để đảm bảo tốc độ nhanh (< 15 phút).


In [ ]:
import os
from hybrid_retriever import HybridSearcher
from evaluator import compute_metrics
from tqdm import tqdm

# Kiểm tra các file cache embeddings có tồn tại và đầy đủ dung lượng (> 5 MB) không
def is_valid_cache(fname, min_mb=5):
    return os.path.exists(fname) and (os.path.getsize(fname) / 1024**2 >= min_mb)

has_all_dense_caches = (
    is_valid_cache("corpus_embeddings_bgem3_resolved.pkl", min_mb=10) and
    is_valid_cache("corpus_embeddings_e5_resolved.pkl", min_mb=10) and
    is_valid_cache("corpus_embeddings_finetuned_resolved.pkl", min_mb=10)
)

# Tự động chọn chế độ an toàn & tối ưu nhất
use_light = not has_all_dense_caches
if has_all_dense_caches:
    print("🔥 Đã phát hiện đầy đủ 3 cache embeddings hợp lệ (BGE-M3 + E5-Large + Bi-Encoder)!")
    print("🚀 Kích hoạt FULL ENSEMBLE MODE (light_mode=False) để tối đa hóa điểm số!")
else:
    print("⚡ Chưa đủ hoặc thiếu cache embeddings. Tự động chạy LIGHT MODE (BM25 + Bi-Encoder).")
    print("   (Chế độ này an toàn tuyệt đối, Recall@5 vẫn đạt ~90% và thời gian chạy cực nhanh!)")

# Khởi tạo hệ thống 2-Stage chuẩn BTC UIT
has_ft_cross = os.path.exists("fine_tuned_vietnamese_cross_encoder")
hybrid = HybridSearcher(corpus, use_reranker=has_ft_cross, light_mode=use_light)

# --- Đánh giá trên 500 câu hỏi Validation ---
val_items = list(train_data.items())[:500]
val_questions = {k: v["question"] for k, v in val_items}
val_truth = {k: v["answer"] for k, v in val_items}

print("\n🔍 Đang đánh giá trên 500 câu hỏi validation...")
val_preds = {}
for qid, question in tqdm(val_questions.items(), desc="2-Stage Validation"):
    val_preds[qid] = hybrid.search(question, top_k=5)

results = compute_metrics(val_preds, val_truth, k=5)
print("\n" + "=" * 50)
print(f"📊 KẾT QUẢ 2-STAGE HYBRID (Light Mode: {use_light}):")
print(f"👉 Recall@5   : {results['Recall'] * 100:.2f}%")
print(f"👉 Precision@5: {results['Precision'] * 100:.2f}%")
print("=" * 50)


## 📦 Tạo file Submission cho Public Test (1000 câu hỏi)


In [ ]:
import json
import zipfile
from tqdm import tqdm

# Dự đoán trên toàn bộ các câu hỏi Public Test
print(f"📦 Đang sinh kết quả cho {len(test_data)} câu hỏi Public Test...")
test_preds = {}
for qid, item in tqdm(test_data.items(), desc="Predicting Public Test"):
    question = item["question"]
    test_preds[qid] = hybrid.search(question, top_k=5)

# Tạo file submission chuẩn format BTC
formatted_preds = {}
for qid, docs in test_preds.items():
    formatted_preds[str(qid)] = {
        "answer": [str(d) for d in docs[:5]]
    }

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(formatted_preds, f, ensure_ascii=False, indent=4)

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.json", arcname="submission.json")

print(f"\n🎉 Đã tạo thành công file nộp bài: 'submission.zip' ({os.path.getsize('submission.zip') / 1024:.1f} KB)")
print(f"📊 Tổng số câu hỏi đã dự đoán: {len(formatted_preds)}")

# Xem thử 3 câu đầu tiên
for i, (qid, pred) in enumerate(list(formatted_preds.items())[:3]):
    q_text = test_data[qid]["question"][:80]
    print(f"\n  [{qid}] {q_text}...")
    print(f"  → Dự đoán: {pred['answer']}")


## 📊 (Tùy chọn) Trực quan hóa Không gian Vector (Embedding Space) bằng t-SNE
> Có thể bỏ qua cell này nếu chỉ cần sinh file submission.


In [ ]:
from visualize_tsne import run_tsne_visualization
from IPython.display import Image, display

run_tsne_visualization(
    model_path="fine_tuned_vietnamese_bi_encoder",
    num_samples=200,
    output_image="embedding_tsne_visualization.png"
)
if os.path.exists("embedding_tsne_visualization.png"):
    display(Image("embedding_tsne_visualization.png", width=900))


## 🔧 (Tùy chọn) Huấn luyện nâng cao: Semi-Hard Negative Mining + Cross-Encoder
> ⚠️ Chỉ chạy nếu muốn huấn luyện lại Cross-Encoder từ đầu (mỗi bước ~15-30 phút GPU).

**Bước 1**: Khai thác mẫu âm bán khó (Semi-Hard Negatives) từ Top 90 Bi-Encoder (Nếu đã có `semi_hard_negatives.pkl` thì tự động bỏ qua).  
**Bước 2**: Huấn luyện PhoRanker Cross-Encoder với BCEWithLogitsLoss (2 epochs) ➔ **Tự động sinh file `submission_finetuned.zip`**!


In [ ]:
# === Bước 1: Khai thác Semi-Hard Negatives (50 mẫu/câu thay vì 10) ===
import os

# ⚠️ QUAN TRỌNG: Phải khai thác LẠI nếu đã tăng num_negatives từ 10 → 50!
# Xóa file cũ để buộc khai thác lại với 50 mẫu âm mỗi câu hỏi.
neg_file = "semi_hard_negatives.pkl"
if os.path.exists(neg_file):
    import pickle
    with open(neg_file, "rb") as f:
        old_negs = pickle.load(f)
    # Kiểm tra số mẫu trung bình để biết cần khai thác lại không
    avg_negs = sum(len(v) for v in old_negs.values()) / max(1, len(old_negs))
    if avg_negs < 20:  # File cũ chỉ có ~10 mẫu/câu
        print(f"⚠️ File '{neg_file}' cũ chỉ có trung bình {avg_negs:.0f} mẫu/câu. Cần khai thác lại với 50 mẫu!")
        os.remove(neg_file)
        from mine_semi_hard_negatives import mine_semi_hard_negatives
        mine_semi_hard_negatives()
    else:
        print(f"✅ '{neg_file}' đã có {avg_negs:.0f} mẫu/câu. Đủ tốt, bỏ qua khai thác lại!")
else:
    from mine_semi_hard_negatives import mine_semi_hard_negatives
    mine_semi_hard_negatives()


In [ ]:
# === Bước 2: Huấn luyện Cross-Encoder (PhoRanker) & Sinh Submission Tối Ưu ===
# 🛠️ ĐÃ SỬA 3 LỖI CỐT LÕI:
#   1. ✅ Đồng bộ Pyvi tokenization giữa Train và Inference (tránh OOV subword)
#   2. ✅ Fusion weights: 0.75 * Stage1 + 0.25 * Reranker (bảo toàn Recall 89.4% của Stage 1)
#   3. ✅ Chỉ re-rank Top 30 ứng viên tinh hoa (thay vì 90, tránh nhiễu từ hạng 50-90)

from train_cross_encoder import train_cross_encoder
import os
import json
import zipfile
from tqdm import tqdm
from hybrid_retriever import HybridSearcher

# 1. Huấn luyện mô hình PhoRanker với Semi-Hard Negatives (2 epochs, 50 mẫu âm/câu)
train_cross_encoder()

# 2. Khởi tạo Retriever với mô hình Cross-Encoder vừa huấn luyện xong!
FT_MODEL_DIR = "fine_tuned_vietnamese_cross_encoder"
if os.path.exists(FT_MODEL_DIR):
    print("\n" + "=" * 65)
    print("🚀 KHỞI ĐỘNG RETRIEVER VỚI 3 FIX CỐT LÕI:")
    print("   1. Pyvi tokenization đồng bộ Train ↔ Inference")
    print("   2. Fusion: 0.75 * Stage1 + 0.25 * Reranker")
    print("   3. Re-rank Top 30 (thay vì 90)")
    print("=" * 65)
    hybrid_ft = HybridSearcher(corpus, use_reranker=True, light_mode=use_light, reranker_model_path=FT_MODEL_DIR)

    # 3. Đánh giá trên 500 câu Validation ĐỂ KIỂM CHỨNG trước khi nộp
    print("\n🔍 Đánh giá trên 500 câu hỏi Validation với mô hình Fine-tuned...")
    val_preds_ft = {}
    for qid, question in tqdm(val_questions.items(), desc="Fine-tuned Validation"):
        val_preds_ft[qid] = hybrid_ft.search(question, top_k=5)
    
    from evaluator import compute_metrics
    results_ft = compute_metrics(val_preds_ft, val_truth, k=5)
    print("\n" + "=" * 50)
    print(f"📊 KẾT QUẢ FINE-TUNED (3 Fix cốt lõi):")
    print(f"👉 Recall@5   : {results_ft['Recall'] * 100:.2f}%")
    print(f"👉 Precision@5: {results_ft['Precision'] * 100:.2f}%")
    print("=" * 50)

    # 4. Dự đoán trên tập Public Test (1000 câu hỏi)
    print(f"\n📦 Đang sinh kết quả cho {len(test_data)} câu hỏi Public Test với mô hình Fine-tuned...")
    test_preds_ft = {}
    for qid, item in tqdm(test_data.items(), desc="Predicting (Fine-tuned PhoRanker)"):
        question = item["question"]
        test_preds_ft[qid] = hybrid_ft.search(question, top_k=5)

    # 5. Định dạng và đóng gói submission_finetuned.zip (bên trong chứa submission.json chuẩn BTC)
    formatted_preds_ft = {}
    for qid, docs in test_preds_ft.items():
        formatted_preds_ft[str(qid)] = {
            "answer": [str(d) for d in docs[:5]]
        }

    with open("submission_finetuned.json", "w", encoding="utf-8") as f:
        json.dump(formatted_preds_ft, f, ensure_ascii=False, indent=4)

    with zipfile.ZipFile("submission_finetuned.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("submission_finetuned.json", arcname="submission.json")

    size_kb = os.path.getsize("submission_finetuned.zip") / 1024
    print("\n" + "=" * 65)
    print(f"🎉 ĐÃ TẠO THÀNH CÔNG: 'submission_finetuned.zip' ({size_kb:.1f} KB)")
    print("👉 Bạn đã có 2 file submission hoàn chỉnh để nộp:")
    print("   1. 'submission.zip' (Từ Cell 8: Stage 1 gốc - Recall ~89.4%)")
    print("   2. 'submission_finetuned.zip' (Từ Cell 13: PhoRanker Fine-tuned + 3 Fix cốt lõi)")
    print("=" * 65)
else:
    print(f"⚠️ Không tìm thấy thư mục mô hình '{FT_MODEL_DIR}'.")
